# 本体 + MTP ヘッドへのマージと、MTP 投機的デコーディング付き vLLM 配信

`install_guide/07_inference_and_merge.md` ステップ3 の Notebook です。

1. 学習ジョブの LoRA アダプタを本体と MTP ヘッドの両方にマージした HF 形式のモデル（`mtp.*` 入り）を、SageMaker Training Job（`src/inference/merge_adapter_mtp.py`）で作る
2. AWS の vLLM DLC で `SM_VLLM_SPECULATIVE_CONFIG`（`method: mtp`）を付けてエンドポイントにデプロイする
3. Chat Completions 形式で呼び出し、ステップ2（MTP なし）の出力と比較する
4. CloudWatch のコンテナログで投機的デコーディングの受理率を確認する
5. エンドポイントを削除する

エンドポイントは起動中ずっと課金されるため、最後の削除セルを必ず実行してください。
vLLM のインストールは不要です（vLLM DLC のコンテナ内で動き、Notebook からは HTTP で呼び出すだけです）。


In [ ]:
import os, json, time, glob, tarfile, boto3, sagemaker
from sagemaker.pytorch import PyTorch
from sagemaker.model import Model
from sagemaker.predictor import Predictor
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

sess    = sagemaker.Session()
try:
    role = os.environ.get('SAGEMAKER_ROLE') or sagemaker.get_execution_role()
except Exception:
    raise SystemExit('ローカル実行時は SAGEMAKER_ROLE 環境変数に SageMaker 実行ロールの ARN を設定してください')
region  = boto3.Session().region_name
account = boto3.client('sts').get_caller_identity()['Account']
bucket  = sess.default_bucket()

train_image_uri = f'{account}.dkr.ecr.{region}.amazonaws.com/nemo-automodel-sagemaker:0.6.0-pt2.10-py313-cu130'
VLLM_IMAGE_TAG = 'server-sagemaker-cuda-v2.5'   # AWS 公式 vLLM DLC (AL2023, vLLM 0.30.0)
vllm_image_uri = f'763104351884.dkr.ecr.{region}.amazonaws.com/vllm:{VLLM_IMAGE_TAG}'
print('sagemaker', sagemaker.__version__, '| region', region, '| bucket', bucket)
print('train image:', train_image_uri)
print('vllm image :', vllm_image_uri)

## 1. マージ対象のアダプタ

`training_job_name` に学習ジョブ名を入れると、アダプタの S3 URI と `model_id` を取り出します。
ステップ2 のマージ済み `model.tar.gz` があれば `merged_s3` に指定すると、本体のマージを省略して MTP ヘッドの追加だけを行います（無ければ本体もこのジョブでマージします）。
**`merged_s3` を使う場合も学習ジョブ（アダプタ）の指定は必要です。** MTP ヘッドに足し込む LoRA の重みはアダプタにしか入っていません。


In [ ]:
training_job_name = ''   # 例: 'automodel-qwen35-cooking-lora-2026-09-24-05-06-27-958'
adapter_s3 = ''          # training_job_name を空にして直接指定してもよい
merged_s3  = ''          # 任意: ステップ2 の model.tar.gz (例: f's3://{bucket}/automodel-merge-adapter-.../output/model.tar.gz')

if training_job_name:
    desc = sess.sagemaker_client.describe_training_job(TrainingJobName=training_job_name)
    adapter_s3 = desc['ModelArtifacts']['S3ModelArtifacts']
    model_id   = json.loads(desc['HyperParameters'].get('model_id', '"Qwen/Qwen3.5-0.8B"'))
else:
    model_id = 'Qwen/Qwen3.5-0.8B'
assert adapter_s3, 'training_job_name か adapter_s3 を指定してください (merged_s3 を使う場合もアダプタは必要: MTP ヘッドの LoRA はアダプタにしか無い)'
val_s3 = f's3://{bucket}/automodel/cooking_basics/validation/'
print('adapter :', adapter_s3); print('merged  :', merged_s3 or '(本体もこのジョブでマージ)'); print('model_id:', model_id)

## 2. MTP ヘッド込みのマージジョブ

学習と同じイメージで `merge_adapter_mtp.py` を実行します。ジョブの中で:

- 本体を `Qwen3_5ForConditionalGeneration` + PEFT でマージ（`merged_s3` があれば省略）
- ベースの safetensors から `mtp.*` を直接読み、アダプタの `mtp.*` の LoRA を `W + (lora_alpha / r) · B · A` で足し込む
- 出力の safetensors に `mtp.*` を追加し、`config.json` に MTP 層数（`mtp_num_hidden_layers`）を残す
- HF で読み直して本体の生成がアダプタ付きと一致することを確認

ログの `[merge-mtp]` 行に、マージした MTP モジュールごとの `|delta|/|W|`（LoRA による相対変化量）が出ます。


In [ ]:
merged_mtp_s3 = ''   # 既に MTP 込みの model.tar.gz があれば S3 URI をここに書く (マージジョブをスキップ)

if not merged_mtp_s3:
    inputs = {'adapter': adapter_s3, 'validation': val_s3}
    if merged_s3:
        inputs['merged'] = merged_s3
    mtp_estimator = PyTorch(
        image_uri=train_image_uri,
        entry_point='merge_adapter_mtp.py',
        source_dir='../src/inference',
        role=role,
        base_job_name='automodel-merge-mtp',
        instance_type='ml.g5.2xlarge',
        instance_count=1,
        volume_size=50,
        max_run=3600,
        hyperparameters={'model_id': model_id, 'dtype': 'bfloat16', 'verify': 1, 'num_samples': 3, 'max_new_tokens': 96},
        environment={'HF_HOME': '/tmp/hf', 'HF_TOKEN': os.environ.get('HF_TOKEN', ''), 'WANDB_MODE': 'disabled'},
    )
    mtp_estimator.fit(inputs, wait=True, logs='All')
    mtp_job = mtp_estimator.latest_training_job.name
    desc = sess.sagemaker_client.describe_training_job(TrainingJobName=mtp_job)
    merged_mtp_s3 = desc['ModelArtifacts']['S3ModelArtifacts']
    print('merge job :', mtp_job)

    # merge_mtp_info.json を取り出して要点を表示
    out_dir = f'artifacts/{mtp_job}'; os.makedirs(out_dir, exist_ok=True)
    # output.tar.gz は model.tar.gz と同じ場所 (.../<job>/output/) にある
    out_s3 = merged_mtp_s3.rsplit('/', 1)[0] + '/output.tar.gz'
    out_bucket, out_key = out_s3[len('s3://'):].split('/', 1)
    boto3.client('s3').download_file(out_bucket, out_key, f'{out_dir}/output.tar.gz')
    with tarfile.open(f'{out_dir}/output.tar.gz') as t:
        t.extractall(out_dir)
    info = json.load(open(f'{out_dir}/merge_mtp_info.json', encoding='utf-8'))
    print('lora      :', info['lora'])
    print('base mtp  :', info['base_mtp_keys'], 'keys |', info['base_mtp_count_field'])
    print('layout    :', info['output_layout'])
    for m in info['mtp_modules_merged']:
        print(f"  {m['hf_key']:<45} {str(m['shape']):<14} |delta|/|W| = {m['delta_rel_norm']:.4f}")
    print('verify mtp:', info.get('verify_mtp_keys'))
    if info.get('verify_body') and 'identical' in info['verify_body'][0]:
        print('verify body: identical', sum(v['identical'] for v in info['verify_body']), '/', len(info['verify_body']))
print('merged+mtp:', merged_mtp_s3)

## 3. MTP 投機的デコーディング付き vLLM DLC エンドポイント

ステップ2 と同じデプロイ手順に `SM_VLLM_SPECULATIVE_CONFIG` を足します。エントリポイントは JSON オブジェクトの値を 1 つの引数として
`--speculative-config '{"method":"mtp","num_speculative_tokens":1}'` に変換します。
vLLM は `model_type: qwen3_5` と `mtp_num_hidden_layers` からドラフトモデル `Qwen3_5MTP` を選び、同じチェックポイントの `mtp.*` を読みます。

Qwen3.5-0.8B の MTP ヘッドは 1 層なので `num_speculative_tokens` は 1 を既定にしています（2 以上にすると同じ層を繰り返し使います）。
在庫不足時の候補インスタンスの切り替えと失敗分の削除はステップ2 と同じです。


In [ ]:
SERVED_NAME = 'qwen35-cooking-lora-mtp'
NUM_SPEC_TOKENS = 1
CANDIDATE_INSTANCES = ['ml.g5.2xlarge', 'ml.g6.2xlarge', 'ml.g5.xlarge', 'ml.g6e.2xlarge']   # 在庫が無ければ次を試す
sm = boto3.client('sagemaker')

vllm_env = {
    'SM_VLLM_SERVED_MODEL_NAME': SERVED_NAME,
    'SM_VLLM_MAX_MODEL_LEN': '4096',
    'SM_VLLM_DTYPE': 'bfloat16',
    'SM_VLLM_GPU_MEMORY_UTILIZATION': '0.85',
    'SM_VLLM_SPECULATIVE_CONFIG': json.dumps({'method': 'mtp', 'num_speculative_tokens': NUM_SPEC_TOKENS}),
}

predictor = None
for inst in CANDIDATE_INSTANCES:
    endpoint_name = f'automodel-vllm-mtp-{int(time.time())}'
    vllm_model = Model(image_uri=vllm_image_uri, model_data=merged_mtp_s3, role=role, predictor_cls=Predictor, env=vllm_env)
    t0 = time.time()
    try:
        predictor = vllm_model.deploy(
            instance_type=inst,
            initial_instance_count=1,
            endpoint_name=endpoint_name,
            inference_ami_version='al2-ami-sagemaker-inference-gpu-3-1',
            container_startup_health_check_timeout=900,
            serializer=JSONSerializer(),
            deserializer=JSONDeserializer(),
        )
        instance_type_used = inst
        print(f'endpoint: {endpoint_name} | {inst} | InService まで {time.time()-t0:.0f}s')
        break
    except Exception as e:
        msg = str(e)
        print(f'{inst}: 失敗 ({msg[:200]}...)')
        for fn, kw in ((sm.delete_endpoint, {'EndpointName': endpoint_name}),
                       (sm.delete_endpoint_config, {'EndpointConfigName': endpoint_name}),
                       (sm.delete_model, {'ModelName': vllm_model.name})):
            try:
                fn(**kw)
            except Exception:
                pass
        if 'InsufficientInstanceCapacity' not in msg and 'ResourceLimitExceeded' not in msg:
            raise   # 在庫・クォータ以外の失敗は CloudWatch のコンテナログ (/aws/sagemaker/Endpoints/<name>) を見る
assert predictor is not None, 'すべての候補で在庫かクォータが足りませんでした。時間をおいて再実行してください'

## 4. 呼び出し

ステップ2 と同じ 5 件のプロンプトを greedy（`temperature: 0`）で生成し、所要時間も記録します。


In [ ]:
val_local = 'artifacts/val.jsonl'
os.makedirs('artifacts', exist_ok=True)
boto3.client('s3').download_file(bucket, 'automodel/cooking_basics/validation/val.jsonl', val_local)
prompts = [json.loads(l) for l in open(val_local, encoding='utf-8') if l.strip()][:5]

results = []
for r in prompts:
    t0 = time.time()
    resp = predictor.predict({
        'model': SERVED_NAME,
        'messages': [{'role': 'user', 'content': r['prompt']}],
        'max_tokens': 128,
        'temperature': 0,
        'chat_template_kwargs': {'enable_thinking': False},
    })
    dt = time.time() - t0
    text = resp['choices'][0]['message']['content']
    usage = resp.get('usage') or {}
    results.append({'prompt': r['prompt'], 'expected': r['output'], 'vllm_mtp': text, 'usage': usage, 'seconds': dt})
    print('-' * 80); print('prompt  :', r['prompt']); print('expected:', r['output'][:120]); print('vllm+mtp:', text)
    print(f"  completion_tokens={usage.get('completion_tokens')} time={dt:.2f}s")
json.dump(results, open(f'artifacts/vllm_mtp_{endpoint_name}.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)

## 5. ステップ2（MTP なし）の出力との比較

`03_merge_and_deploy_vllm.ipynb` で保存した `artifacts/vllm_automodel-vllm-*.json` があれば並べます。
投機的デコーディングは greedy では本体と同じトークン列を出す方式なので、一致するのが期待値です（bf16 の演算順の違いで途中から分かれることはあります）。


In [ ]:
cands = sorted(p for p in glob.glob('artifacts/vllm_automodel-vllm-*.json') if '/vllm_mtp_' not in p)
if cands:
    ref = {r['prompt']: r for r in json.load(open(cands[-1], encoding='utf-8'))}
    same = 0
    for r in results:
        base = ref.get(r['prompt'])
        if base is None:
            continue
        same += base['vllm'] == r['vllm_mtp']
        print('-' * 80); print('prompt   :', r['prompt'])
        print('MTP なし :', base['vllm'][:160]); print('MTP あり :', r['vllm_mtp'][:160])
        bu, mu = base.get('usage') or {}, r['usage']
        print(f"  tokens: {bu.get('completion_tokens')} -> {mu.get('completion_tokens')} | 一致: {base['vllm'] == r['vllm_mtp']}")
    print(f'\n完全一致: {same}/{len(results)}  (比較元: {cands[-1]})')
else:
    print('ステップ2 の結果ファイルが見つかりません (比較をスキップ)')

## 6. 受理率の確認（CloudWatch のコンテナログ）

vLLM は投機的デコーディングの統計を `SpecDecoding metrics: Mean acceptance length ...` の形で定期的にログに出します。
エンドポイントのコンテナログは CloudWatch のロググループ `/aws/sagemaker/Endpoints/<endpoint_name>` に入るので、そこから拾います。
あわせて、ドラフトモデルとして `Qwen3_5MTP` が読み込まれた行も探します。統計行が無い場合は、もう一度セル 4 を実行して 30 秒ほど待ってから再実行してください。


In [ ]:
logs = boto3.client('logs')
group = f'/aws/sagemaker/Endpoints/{endpoint_name}'
found = []
token = None
while True:
    kw = dict(logGroupName=group, filterPattern='?SpecDecoding ?Qwen3_5MTP ?speculative ?Speculative')
    if token:
        kw['nextToken'] = token
    resp = logs.filter_log_events(**kw)
    found += [e['message'].rstrip() for e in resp.get('events', [])]
    token = resp.get('nextToken')
    if not token:
        break
print(f'{len(found)} 行')
for line in found[-25:]:
    print(line[:300])

## 7. 後片付け（必ず実行）

In [ ]:
predictor.delete_endpoint(delete_endpoint_config=True)
vllm_model.delete_model()
print('deleted endpoint:', endpoint_name)